In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


In [0]:
def build_store_sales_summary(df):
"""
Filters valid completed transactions, calculates total price,
and aggregates sales metrics per store.
"""
# Filter: guard against nulls explicitly, not just falsy values
filtered_df = df.filter(
F.col("quantity").isNotNull()
& (F.col("quantity") > 0)
& F.col("status").isNotNull()
& (F.col("status") == "completed")
)
# Calculated column: cast to Decimal for currency precision
calc_df = filtered_df.withColumn(
"total_price",
(F.col("quantity") * F.col("unit_price")).cast(DecimalType(12, 2))
)
# Aggregation with null-safe rounding
result_df = (
calc_df.groupBy("store_id")
.agg(
F.round(F.sum("total_price"), 2).alias("total_sales"),
F.round(F.avg("total_price"), 2).alias("avg_order_value"),
F.count("*").alias("num_transactions")
)
.orderBy(F.desc("total_sales"))
)
return result_df
df = spark.table("raw_sales_transactions")
result_df = build_store_sales_summary(df)
result_df.show(truncate=False)